# **Proyecto 05: Sistema de Identificación del Género de una Canción**

---

## **Parte 1: Carga de los datos**

En este proyecto, vamos a construir una red neuronal artificial que puede identificar el género de una canción, utilizando la librería GTZAN Genre Collection. Para extraer los features (características) de las canciones, vamos a utilizar la librería de Python [**librosa**](https://librosa.org/). Utilizaremos los Mel-frequency cepstral coefficients (MFCC), que simulan la escucha humana y son coúnmente utilizados en aplicaciones de speech recognition (reconocimiento del habla) así como en detección del género musical. Estos valores serán los que utilizaremos como entrada de la red neuronal.

Para entender qué son los MFCC, podemos descargar Kick Loop 5 by Stereo Surgeon desde [https://freesound.org/people/Stereo%20Surgeon/sounds/266093](https://freesound.org/people/Stereo%20Surgeon/sounds/266093), y también descargar Whistling by cmagar desde [https://freesound.org/people/grrlrighter/sounds/98195/](https://freesound.org/people/grrlrighter/sounds/98195/). Uno de ellos es un beat de bajas frecuencias, mientras que el otros es un silbido de frecuencias más agudas. Veremos mediante los valores MFCC como estos dos sonidos son claramente distintos.

Además de importar la librería librosa, también vamos a usar [**glob**](https://docs.python.org/es/3/library/glob.html) para poder hacer un listado de los archivos en los diferentes repositorios de los géneros musicales. Además, utilizaremos <span style="color:blue"> **numpy**</span> y <span style="color:blue"> **matplotlib**</span>.

También vamos a importar el modelo **Sequential** de **Keras**. Este es un modelo típico de red neuronal feed-forward. Finalmente, importaremos una capa densa de una red neuronal (capa con una colección de neuronas en ella, [**Dense**](https://keras.io/api/layers/core_layers/dense/https://keras.io/api/layers/core_layers/dense/)).


A diferencia de con las operaciones convolución, por ejemplo, esta red va a tener representaciones en dos dimensiones. Vamos a importar funciones de activación, lo que nos permitirá decidir en cada capa de la red neuronal que función no lineal utilizar, y también importaremos la función <span style="color:blue"> **to_categorical**</span>, que nos permite cambiar los nombres de las clases en categorías, que es lo que ocurre con el one-hot encoding.

En primer lugar, vamos a definir una función llamada <span style="color:blue"> **display_mfcc**</span> que nos permitirá visualizar en un gráfico estos valores. Los MFCC se pueden obtener a partir de la librería librosa. Para visualizar, utilizaremos un tipo de gráfico conocido como espectrograma, que pertenece a la libraría librosa (<span style="color:blue"> **specshow**</span>).

In [ ]:
import librosa
import librosa.display
import glob
import numpy as np
import matplotlib.pyplot as plt

# Importaciones de Keras para la Red Neuronal
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation
from tensorflow.keras.utils import to_categorical

def display_mfcc(file_path, title):
    # 1. Cargamos el archivo de audio
    # y = la señal de audio, sr = sample rate (frecuencia de muestreo)
    y, sr = librosa.load(file_path)
    
    # 2. Extraemos los MFCCs (por defecto suele extraer 20)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    
    # 3. Visualización con specshow
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(mfccs, x_axis='time', sr=sr)
    plt.colorbar(format='%+2.0f dB')
    plt.title(f'MFCC Spectrogram: {title}')
    plt.tight_layout()
    plt.show()
    
    return mfccs

Vamos a probar esta función que hemos creado con las canciones 'kick-loop.wav' y 'whistling.wav' que hemos importado previamente.

In [ ]:
if __main__ == "__name__":
    # Asumiendo que has descargado los archivos y están en tu carpeta:
    mfcc_kick = display_mfcc('kick_loop.wav', 'Kick Loop (Bajos)')
    mfcc_whistle = display_mfcc('whistling.wav', 'Whistling (Agudos)')

---
## **Parte 2: Preprocesado de los datos**

Ahora que ya podemos visualizar los coeficientes MCFF, vamos a crear otra función auxiliar, que nos permitirá obtener dichos coeficientes y guardarlos en un vector. Para mejor funcionamiento de la posterior red neuronal, vamos a normalizar los coeficients para que su rango esté comprendido entre -1 y 1, y solamente vamos a utilizar los primeros 25000 coeficientes. A la función auxiliar la vamos a llamar  <span style="color:blue"> **extract_features_song**</span>

In [1]:
import librosa
import numpy as np

def extract_features_song(ruta_cancion):
    try:
        audio_data, sampling_rate = librosa.load(ruta_cancion)
        # Utilizando un número fijo de coeficientes MFCC (por ejemplo, 40) basado en la práctica común.
        mfcc_features = librosa.feature.mfcc(y=audio_data, sr=sampling_rate, n_mfcc=40)
        # Rellenar o truncar a un tamaño fijo (25000 como se menciona en el texto).
        longitud_maxima_mfcc = 25000
        if mfcc_features.shape[1] < longitud_maxima_mfcc:
            ancho_relleno = longitud_maxima_mfcc - mfcc_features.shape[1]
            mfcc_features = np.pad(mfcc_features, pad_width=((0, 0), (0, ancho_relleno)), mode='constant')
        else:
            mfcc_features = mfcc_features[:, :longitud_maxima_mfcc]

        # Normalizar los coeficientes MFCC para que estén entre -1 y 1.
        mfcc_normalizados = 2 * ((mfcc_features - mfcc_features.min()) / (mfcc_features.max() - mfcc_features.min())) - 1
        return mfcc_normalizados
    except Exception as e:
        print(f"Error al procesar {ruta_cancion}: {e}")
        return None

A continuación vamos a definir una rutina para abrir los ficheros con las canciones de los distintos géneros y extraer los coeficientes MCFF de ellas. La función se llamará  <span style="color:blue"> **generate_features_and_labels**</span>, y en ella haremos un bucle sobre todos los géneros, y buscaremos en la carpeta de cada género todos los archivos, los abriremos y extraeremos los MCFF mediante la función extract_features_song. Al mismo tiempo, crearemos un vector de etiquetas (labels) y añadiremos el género a ese vector cada vez que abramos un fichero. Sin embargo, la red neuronal no es capaz de predecir una palabra o letras. Para poder utilizarla, necesitamos hacer un one-hot encoding, lo que significa que cada palabra será representada por un vector de diez números binarios (tenemos 10 clases en total), de manera que solamente uno de los números sea 1 y el resto sean 0. Utilizaremos la función  <span style="color:blue"> **np.unique**</span> para hacer que las etiquetas se conviertan en números enteros. Luego, utilizamos la función  <span style="color:blue"> **to_categorical**</span>, que convertirá esos números enteros en representación one-hot encoding. Por último, utilizamos la función  <span style="color:blue"> **np.stack**</span> sobre los features que hemos extraído de las canciones para que se junten en una única matriz.    

In [2]:
import glob
import os
import numpy as np
from keras.utils import to_categorical

def generate_features_and_labels(ruta_datos='Data/genres_original'):
    todas_las_caracteristicas = []
    todas_las_etiquetas = []
    generos = sorted(os.listdir(ruta_datos))

    # Iterar a través de cada género musical
    for i, genero in enumerate(generos):
        ruta_genero = os.path.join(ruta_datos, genero)
        # Saltar si no es un directorio (ej., .DS_Store)
        if not os.path.isdir(ruta_genero):
            continue

        print(f"Procesando género: {genero}")
        # Buscar todos los archivos .wav en la carpeta del género
        for archivo_cancion in glob.glob(os.path.join(ruta_genero, '*.wav')):
            # Extraer características MFCC de la canción
            mfcc_extraidos = extract_features_song(archivo_cancion)
            if mfcc_extraidos is not None:
                todas_las_caracteristicas.append(mfcc_extraidos)
                todas_las_etiquetas.append(genero)

    # Convertir etiquetas de texto a números enteros
    etiquetas_unicas = np.unique(todas_las_etiquetas)
    etiqueta_a_entero = {etiqueta: i for i, etiqueta in enumerate(etiquetas_unicas)}
    etiquetas_enteras = np.array([etiqueta_a_entero[etiqueta] for etiqueta in todas_las_etiquetas])

    # Aplicar One-hot encoding a las etiquetas numéricas
    etiquetas_one_hot = to_categorical(etiquetas_enteras, num_classes=len(etiquetas_unicas))

    # Apilar todas las características extraídas en una única matriz NumPy
    # La función 'extract_features_song' ya se encarga de asegurar que todas las características tengan la misma forma
    caracteristicas_finales = np.stack(todas_las_caracteristicas)

    return caracteristicas_finales, etiquetas_one_hot, etiquetas_unicas

In [ ]:
# Generar las características y etiquetas llamando a la función definida anteriormente
# Asegúrate de que la ruta 'Data/genres_original' sea correcta en tu entorno
caracteristicas_finales, etiquetas_one_hot, etiquetas_unicas = generate_features_and_labels('Data/genres_original')

Vamos a dividir el dataset de features y labels en train y test, dejando un 80% de los datos para el train y un 20% para el test. Antes de dividir, hay que primero juntar los datos de los features y las etiquetas utilizando la función  <span style="color:blue"> **np.column_stack**</span>, luego  hay que barajar los datos para no producir artefactos debido al orden de los datos en el modelo, para ello, podemos utilizar la función  <span style="color:blue"> **shuffle**</span>. Una vez separados en train y test, debemos eliminar de cada set las 10 últimas columnas, puesto que son las columnas correspondientes al one-hot encoding, que no necesitamos como entrada, si no como salida de nuestra red neuronal.

In [ ]:
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

# Asumiendo que ya tienes 'caracteristicas_finales' y 'etiquetas_one_hot' de la función anterior
# Aplanamos las características si es necesario para el column_stack
n_muestras, n_filas, n_cols = caracteristicas_finales.shape
caracteristicas_planas = caracteristicas_finales.reshape(n_muestras, -1)

# Combinar características y etiquetas
datos_combinados = np.column_stack((caracteristicas_planas, etiquetas_one_hot))

# Barajar los datos
datos_barajados = shuffle(datos_combinados)

# Dividir en train (80%) y test (20%)
train_data, test_data = train_test_split(datos_barajados, test_size=0.2, random_state=42)

# Separar las características de las etiquetas (las últimas 10 columnas son el one-hot encoding)
X_train = train_data[:, :-10]
y_train = train_data[:, -10:]

X_test = test_data[:, :-10]
y_test = test_data[:, -10:]

# Redimensionar X para que coincida con la entrada esperada de la red si es necesario
# (En este caso, lo mantenemos plano para una capa Dense inicial)
print(f"Forma de X_train: {X_train.shape}")
print(f"Forma de y_train: {y_train.shape}")

---
## **Parte 3: Análisis del modelo (entrenamiento de la red neuronal)**

Seguidamente construimos la red neuronal. Vamos a utilizar un modelo secuencial ( <span style="color:blue"> **Sequential**</span>). Añadiremos una capa densa de 100 neuronas en primer lugar, definiendo también en esta primera capa el tamaño de entrada de la red (que podemos obtener a partir del tamaño del train dataset). La función de activación que escogeremos para la primera capa es la función  <span style="color:red"> 'relu'</span>, y luego añadimos una segunda capa densa de 10 neuronas con una función de activación de tipo  <span style="color:red"> 'softmax'</span>. El tamaño de la segunda capa viene fijado por el número de clases que tenemos en nuestro clasificador, es decir 10 géneros musicales distintos. La activación  <span style="color:red"> 'softmax'</span> coge las 10 salidas y las normaliza de tal forma que la suma de las 10 sea 1. De esta forma, acaban representando probabilidades, así que la predicción será aquella que tenga una probabilidad mayor.

Una vez creado el modelo, debemos compilarlo (instrucción [**compile**](https://www.tensorflow.org/api_docs/python/tf/keras/Model)), añadiendo algunos hiperparámetros de como funciona nuestro algoritmo. En concreto, vamos a utilizar el optimizador  <span style="color:red"> 'adam'</span>, y la función de pérdidas será  <span style="color:red"> 'categorical_crossentropy'</span>. Por último, la medida que queremos ver durante la evaluación es la medida de  <span style="color:red"> 'accuracy'</span>. Par apoder ver un resumen de la arquitectura creada, podemos mostrar por pantalla un resumen <span style="color:blue"> **model.summary**</span> que nos da los detalles sobre las capas.

In [1]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
# Supongamos que ya tienes tus datos cargados: X_train, y_train, X_test, y_test
# El tamaño de entrada se obtiene de las columnas de tu dataset (X_train.shape[1])
input_dim = X_train.shape[1] 
# --- 1. Construcción del modelo ---
model = Sequential()
# Primera capa densa: 100 neuronas, activación 'relu' y tamaño de entrada
model.add(Dense(100, activation='relu', input_shape=(input_dim,)))
# Segunda capa densa: 10 neuronas (para los 10 géneros), activación 'softmax'
model.add(Dense(10, activation='softmax'))
# --- 2. Compilación del modelo ---
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy', # Asegúrate de que tus etiquetas estén en formato one-hot
    metrics=['accuracy']
)
# Mostrar el resumen de la arquitectura
model.summary()
# --- 3. Entrenamiento del modelo ---
history = model.fit(
    X_train, 
    y_train, 
    epochs=10, 
    batch_size=32, 
    validation_split=0.2
)
# --- 4. Evaluación del sistema ---
loss, accuracy = model.evaluate(X_test, y_test)
print(f"\nPrecisión en el conjunto de test: {accuracy:.4f}")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


I0000 00:00:1777459859.662771   10379 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777459863.746732   10379 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


NameError: name 'X_train' is not defined

Por último, solamente tenemos que entrenar el model [**fit**](https://www.tensorflow.org/api_docs/python/tf/keras/Model) definiendo como entradas los datos de entrenamiento (tanto los features como las etiquetas, y definimos el número de épocas (10 por ejemplo), el tamaño del batch (32 por ejemplo) y el tamaño del conjunto de validación (0.2 por ejemplo). Por último, obtendremos las métricas de nuestro sistema evaluando la precisión mediante los datos de test con la función [**evaluate**](https://www.tensorflow.org/api_docs/python/tf/keras/Model).